# Módulo 05 — Juegos de Suma Cero y Teorema Minimax

**Objetivos**: Resolver juegos de suma cero con el algoritmo minimax (estrategias puras) y con `scipy.optimize.linprog` para estrategias mixtas.

In [ ]:
import numpy as np
from scipy.optimize import linprog
import matplotlib.pyplot as plt
print('Librerías cargadas.')

## 1. Minimax en estrategias puras

El algoritmo:
1. J1 calcula **maximin**: max sobre filas del mínimo de cada fila.
2. J2 calcula **minimax**: min sobre columnas del máximo de cada columna.
3. Si maximin = minimax → existe **punto de silla** (equilibrio puro).

In [ ]:
def solve_minimax(A):
    """
    A: matriz de pagos de J1 (juego de suma cero, J2 recibe -A)
    Devuelve: (maximin, minimax, saddle_point or None)
    """
    row_mins = A.min(axis=1)
    col_maxs = A.max(axis=0)
    maximin = row_mins.max()
    minimax = col_maxs.min()
    i_star = row_mins.argmax()
    j_star = col_maxs.argmin()

    saddle = None
    if np.isclose(maximin, minimax):
        saddle = (int(i_star), int(j_star))

    return maximin, minimax, saddle

# Ejemplo con punto de silla
A1 = np.array([[3, -1, 0],
               [-2,  2, 1],
               [ 1,  0,-1]])

maximin, minimax, saddle = solve_minimax(A1)
print(f'Maximin: {maximin}, Minimax: {minimax}')
if saddle:
    print(f'Punto de silla en ({saddle[0]+1}, {saddle[1]+1}), valor = {A1[saddle]}')
else:
    print('No existe punto de silla puro — requiere estrategias mixtas')

## 2. Resolver con linprog (estrategias mixtas)

Para juegos sin punto de silla, el valor del juego y las estrategias mixtas óptimas se obtienen resolviendo un programa lineal.

**Problema de J1** (maximizar el valor garantizado v):
- max v s.t. Aᵀ·p ≥ v·1, Σpᵢ=1, pᵢ≥0

Transformamos a forma estándar para `linprog`.

In [ ]:
def solve_zero_sum_lp(A):
    """
    Resuelve un juego de suma cero con linprog.
    Devuelve: (p_opt, q_opt, value)
      p_opt: estrategia mixta óptima de J1 (distribución sobre filas)
      q_opt: estrategia mixta óptima de J2 (distribución sobre columnas)
      value: valor del juego para J1
    """
    m, n = A.shape
    # Trasladamos para que A sea positiva (no cambia las estrategias)
    shift = -A.min() + 1
    A_pos = A + shift

    # Problema de J2: min Σqⱼ s.t. A·q ≤ 1·Σqⱼ → equivalente a min 1ᵀy s.t. A·y ≥ 1, y≥0
    # Usamos la formulación estándar: min cᵀy s.t. Aub·y ≤ bub, Aeq·y = beq
    c = np.ones(n)
    A_ub = -A_pos.T  # (n, m) → queremos A·y >= 1, o sea -A·y <= -1
    b_ub = -np.ones(m)
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=[(0, None)]*n, method='highs')

    if not res.success:
        return None, None, None

    y = res.x
    sum_y = y.sum()
    q_opt = y / sum_y
    v_shifted = 1.0 / sum_y
    value = v_shifted - shift

    # Calcular p_opt: los multiplicadores del dual
    # (alternativamente, resolver el PL primal para J1)
    c2 = -np.ones(m)
    A_ub2 = A_pos  # A·p <= v·1 para J2 → -A·p >= -v·1
    A_ub2 = -A_pos
    b_ub2 = -np.ones(n)
    res2 = linprog(c2, A_ub=A_ub2, b_ub=b_ub2, bounds=[(0, None)]*m, method='highs')
    if res2.success:
        x2 = res2.x
        p_opt = x2 / x2.sum()
    else:
        p_opt = np.ones(m) / m

    return p_opt, q_opt, value

# Juego sin punto de silla (Matching Pennies)
A_mp = np.array([[1, -1], [-1, 1]])
p, q, v = solve_zero_sum_lp(A_mp)
print(f'Matching Pennies:')
print(f'  p* (J1) = {np.round(p, 4)}')
print(f'  q* (J2) = {np.round(q, 4)}')
print(f'  Valor del juego = {v:.4f} (esperado: 0)')

In [ ]:
# Juego de póker simplificado
A_poker = np.array([[ 0, -1,  2],
                    [ 1,  0, -1],
                    [-2,  1,  0]])
p, q, v = solve_zero_sum_lp(A_poker)
print(f'Póker simplificado:')
print(f'  p* (J1) = {np.round(p, 3)}')
print(f'  q* (J2) = {np.round(q, 3)}')
print(f'  Valor del juego = {v:.4f}')

## 3. Visualización: pago garantizado según estrategia mixta

Para un juego 2×2, mostramos el pago mínimo garantizable de J1 en función de su mezcla p.

In [ ]:
A_2 = np.array([[3, 1], [0, 4]])
ps = np.linspace(0, 1, 200)

# Para cada p, el pago mínimo que puede garantizar J1 es min sobre columnas de J2
min_payoff = np.array([min(A_2[0,j]*p + A_2[1,j]*(1-p) for j in range(2)) for p in ps])

_, _, v_opt = solve_zero_sum_lp(A_2)

plt.figure(figsize=(7, 4))
plt.plot(ps, min_payoff, color='#1a3a5c', lw=2.5, label='Pago mínimo garantizado por J1')
plt.axhline(v_opt, color='#2d7a50', ls='--', lw=1.5, label=f'Valor del juego = {v_opt:.3f}')
plt.xlabel('p (prob. J1 de jugar fila 1)')
plt.ylabel('Pago garantizado')
plt.title('Pago mínimo garantizable vs. mezcla de J1')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Ejercicios

**Ejercicio 1**: Comprueba que en el juego de guerra de precios (A = [[-1,-3],[0,-2]]) existe punto de silla. ¿Cuál es la estrategia dominante?

**Ejercicio 2**: Usa `solve_zero_sum_lp` para resolver el juego de piedra-papel-tijera y verifica que el valor es 0 y la estrategia óptima es 1/3 para cada opción.

In [ ]:
# Tu código aquí
